# 100-Gene gLM vs One-Hot vs VCF: Model-Wise Analysis (Kaggle)

This notebook expects two Kaggle datasets:
1. 100_genes_embeddings (same as before)
2. 100-genes-encodings (contains Encodings and VCF)

It computes the same 5 analyses from the 7-gene workflow, but plots them model-wise:
- X-axis: genes (sorted by sequence length extracted from gene name suffix after underscore)
- Y-axis: metric value

The 5 analyses are:
1. Individual ARI
2. Individual silhouette
3. Pairwise ARI
4. CKA (gLM vs one-hot)
5. CKA (gLM vs VCF)

Finally, it creates model-comparison averages across all genes for each analysis.

In [ ]:
import ast
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

sns.set_theme(style="whitegrid")

In [ ]:
# -----------------------------
# Dataset paths and run config
# -----------------------------
EMBED_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/mruhaib/100-genes-embeddings/100_genes_embeddings"),
    Path("/kaggle/input/datasets/mruhaib/100_genes_embeddings/100_genes_embeddings"),
    Path("/kaggle/input/100-genes-embeddings/100_genes_embeddings"),
    Path("/kaggle/input/100_genes_embeddings/100_genes_embeddings"),
    Path("/kaggle/input/100-genes-embeddings"),
    Path("/kaggle/input/100_genes_embeddings"),
    Path("/kaggle/input/datasets/mruhaib/100-genes-embeddings"),
    Path("/kaggle/input/datasets/mruhaib/100_genes_embeddings"),
]

ENCODINGS_DATASET_CANDIDATES = [
    Path("/kaggle/input/100-genes-encodings"),
    Path("/kaggle/input/100_genes_encodings"),
    Path("/kaggle/input/datasets/mruhaib/100-genes-encodings"),
    Path("/kaggle/input/datasets/mruhaib/100_genes_encodings"),
]

WORK_ROOT = Path("/kaggle/working/100_gene_modelwise_eval")
RESULTS_DIR = WORK_ROOT / "Results"
PLOTS_DIR = RESULTS_DIR / "modelwise_plots"

K_CLUSTERS = 30
SHOW_PLOTS = False

for p in [WORK_ROOT, RESULTS_DIR, PLOTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def first_existing_dir(candidates):
    for c in candidates:
        if c.exists() and c.is_dir():
            return c
    return None

def first_existing_file(candidates):
    for c in candidates:
        if c.exists() and c.is_file():
            return c
    return None

def find_named_subdir(root, target_name):
    direct = root / target_name
    if direct.exists() and direct.is_dir():
        return direct

    nested = root / "100_gene_eval" / target_name
    if nested.exists() and nested.is_dir():
        return nested

    matches = sorted([p for p in root.rglob(target_name) if p.is_dir()])
    return matches[0] if matches else None

def autodetect_encodings_root():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for p in sorted(input_root.iterdir()):
        if not p.is_dir():
            continue
        has_enc = (p / "Encodings").exists() or (p / "100_gene_eval" / "Encodings").exists()
        has_vcf = (p / "VCF").exists() or (p / "100_gene_eval" / "VCF").exists()
        if has_enc and has_vcf:
            return p
    return None

def embedding_root_score(root):
    if root is None or (not root.exists()) or (not root.is_dir()):
        return -1

    model_dirs = [d for d in root.iterdir() if d.is_dir()]
    direct_model_txt_hits = 0
    for d in model_dirs[:80]:
        if any(d.glob("*.txt")):
            direct_model_txt_hits += 1

    nested_embedding_hits = len(list(root.glob("*/*_embeddings.txt")))
    direct_txt_hits = len(list(root.glob("*.txt")))

    return (5 * direct_model_txt_hits) + nested_embedding_hits + direct_txt_hits

def normalize_embed_root(path):
    if path is None:
        return None

    sub = path / "100_genes_embeddings"
    if sub.exists() and sub.is_dir():
        if embedding_root_score(sub) >= embedding_root_score(path):
            return sub

    return path

def autodetect_embeddings_root():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None

    candidates = []
    for p in input_root.iterdir():
        if not p.is_dir():
            continue
        candidates.append(p)
        for sub in p.iterdir():
            if sub.is_dir():
                candidates.append(sub)

    best = None
    best_score = -1
    for c in candidates:
        score = embedding_root_score(c)
        if score > best_score:
            best_score = score
            best = c

    if best_score <= 0:
        return None
    return normalize_embed_root(best)

EMBED_ROOT = first_existing_dir(EMBED_ROOT_CANDIDATES)
if EMBED_ROOT is not None:
    EMBED_ROOT = normalize_embed_root(EMBED_ROOT)
else:
    EMBED_ROOT = autodetect_embeddings_root()

if EMBED_ROOT is None:
    raise FileNotFoundError("Could not locate 100_genes_embeddings dataset root.")

ENC_DATASET_ROOT = first_existing_dir(ENCODINGS_DATASET_CANDIDATES)
if ENC_DATASET_ROOT is None:
    ENC_DATASET_ROOT = autodetect_encodings_root()
if ENC_DATASET_ROOT is None:
    raise FileNotFoundError("Could not locate 100-genes-encodings dataset root.")

ENC_DIR = find_named_subdir(ENC_DATASET_ROOT, "Encodings")
VCF_DIR = find_named_subdir(ENC_DATASET_ROOT, "VCF")
if ENC_DIR is None:
    raise FileNotFoundError("Could not find Encodings folder inside 100-genes-encodings dataset.")

META_CANDIDATES = [
    ENC_DATASET_ROOT / "metadata2.csv",
    ENC_DATASET_ROOT / "100_gene_eval" / "metadata2.csv",
    Path("/kaggle/input/datasets/mruhaib/100_genes/metadata2.csv"),
    Path("/kaggle/input/100-genes/metadata2.csv"),
    Path("/kaggle/input/metadata2/metadata2.csv"),
]
META_PATH = first_existing_file(META_CANDIDATES)

embed_model_dirs = sorted([d.name for d in EMBED_ROOT.iterdir() if d.is_dir()])

print(f"EMBED_ROOT      : {EMBED_ROOT}")
print(f"EMBED model dirs: {len(embed_model_dirs)}")
print(f"EMBED sample    : {embed_model_dirs[:8]}")
print(f"ENC_DATASET_ROOT: {ENC_DATASET_ROOT}")
print(f"ENC_DIR         : {ENC_DIR}")
print(f"VCF_DIR         : {VCF_DIR if VCF_DIR is not None else 'NOT FOUND'}")
print(f"metadata2.csv   : {META_PATH if META_PATH is not None else 'NOT FOUND (ARI-vs-clades may be NaN)'}")
print(f"RESULTS_DIR      : {RESULTS_DIR}")

In [ ]:
# -----------------------------
# Helpers
# -----------------------------
def extract_strain_name(sample_id, gene):
    marker = f"_{gene}_"
    idx = sample_id.find(marker)
    return sample_id[:idx] if idx >= 0 else sample_id

def extract_gene_length(gene_name):
    m = re.search(r"_(\d+)$", str(gene_name))
    return int(m.group(1)) if m else np.nan

def safe_filename(text):
    return "".join(ch if (ch.isalnum() or ch in "-_.") else "_" for ch in str(text))

def load_glm_embeddings(path):
    """
    Supports both embedding file formats:
    1) list of dicts with key 'embedding'
    2) plain list of embeddings (one row per strain)
    """
    obj = ast.literal_eval(path.read_text(encoding="utf-8"))

    if isinstance(obj, dict):
        if "embeddings" in obj:
            obj = obj["embeddings"]
        elif "data" in obj:
            obj = obj["data"]
        else:
            raise ValueError(f"Unsupported dict embedding format in {path}")

    if not isinstance(obj, (list, tuple)):
        raise ValueError(f"Unsupported embedding container type in {path}: {type(obj)}")

    if len(obj) == 0:
        return np.zeros((0, 0), dtype=float)

    first = obj[0]
    if isinstance(first, dict):
        if "id" in first:
            try:
                ids = [int(item["id"]) for item in obj]
                if ids == list(range(1, len(ids) + 1)):
                    obj = sorted(obj, key=lambda x: int(x["id"]))
            except Exception:
                pass

        vectors = []
        for item in obj:
            if "embedding" in item:
                vectors.append(item["embedding"])
            elif "vector" in item:
                vectors.append(item["vector"])
            else:
                raise ValueError(f"Missing 'embedding' key in dict item for {path}")

        arr = np.asarray(vectors, dtype=float)
    else:
        # Plain list-of-embeddings format (your 1011-strain files).
        arr = np.asarray(obj, dtype=float)

    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    if arr.ndim != 2:
        raise ValueError(f"Expected 2D embeddings matrix in {path}, got shape {arr.shape}")

    return arr

def infer_model_from_stem(stem, gene, fallback):
    # Supports stems like:
    # - model_gene
    # - model_gene_embeddings
    # - gene
    # - gene_embeddings
    core = stem
    if core.endswith("_embeddings"):
        core = core[: -len("_embeddings")]

    if core == gene:
        return fallback

    suffix = f"_{gene}"
    if core.endswith(suffix):
        model = core[: -len(suffix)]
        if model:
            return model

    return fallback

def first_existing_file(paths):
    for p in paths:
        if p.exists() and p.is_file():
            return p
    return None

def candidate_paths_in_model_dir(model_dir, gene):
    return [
        model_dir / f"{gene}_embeddings.txt",
        model_dir / f"{gene}.txt",
        model_dir / f"{model_dir.name}_{gene}_embeddings.txt",
        model_dir / f"{model_dir.name}_{gene}.txt",
    ]

def find_embedding_files_for_gene(embed_root, gene):
    """
    Supports these layouts:
    1) gene-first: EMBED_ROOT/<gene>/<model>_<gene>.txt
    2) model-first: EMBED_ROOT/<model>/<gene>_embeddings.txt
    3) model-first: EMBED_ROOT/<model>/<model>_<gene>_embeddings.txt
    4) model-first legacy: EMBED_ROOT/<model>/<gene>.txt or <model>_<gene>.txt
    """
    out = {}

    # Layout 1: gene-first
    gene_dir = embed_root / gene
    if gene_dir.exists() and gene_dir.is_dir():
        for p in sorted(gene_dir.glob("*.txt")):
            model_name = infer_model_from_stem(p.stem, gene, p.stem)
            out[model_name] = p

    # Layout 2/3/4: model-first
    model_dirs = sorted([d for d in embed_root.iterdir() if d.is_dir() and d.name != gene])
    for model_dir in model_dirs:
        hit = first_existing_file(candidate_paths_in_model_dir(model_dir, gene))

        if hit is None:
            patterns = [
                f"{gene}_embeddings.txt",
                f"{gene}.txt",
                f"*_{gene}_embeddings.txt",
                f"*_{gene}.txt",
                f"*{gene}*_embeddings.txt",
                f"*{gene}*.txt",
            ]
            for pat in patterns:
                matches = sorted(model_dir.glob(pat))
                if not matches:
                    matches = sorted(model_dir.rglob(pat))
                if matches:
                    hit = matches[0]
                    break

        if hit is not None:
            model_name = infer_model_from_stem(hit.stem, gene, model_dir.name)
            out[model_name] = hit

    # Broad fallback if still empty
    if len(out) == 0:
        patterns = [
            f"{gene}_embeddings.txt",
            f"{gene}.txt",
            f"*_{gene}_embeddings.txt",
            f"*_{gene}.txt",
            f"*{gene}*_embeddings.txt",
            f"*{gene}*.txt",
        ]
        seen = set()
        for pat in patterns:
            for p in sorted(embed_root.rglob(pat)):
                key = str(p)
                if key in seen:
                    continue
                seen.add(key)
                model_name = infer_model_from_stem(p.stem, gene, p.parent.name)
                out[model_name] = p

    return dict(sorted(out.items()))

def load_clade_labels_for_gene(sample_ids, gene):
    """
    Tries metadata2.csv first. If unavailable, accepts a row-aligned labels.txt file.
    """
    metadata_candidates = [
        ENC_DATASET_ROOT / "metadata2.csv",
        ENC_DATASET_ROOT / "100_gene_eval" / "metadata2.csv",
        EMBED_ROOT.parent / "metadata2.csv",
        EMBED_ROOT.parent.parent / "Data Labelling" / "metadata2.csv",
        Path("/kaggle/input/datasets/mruhaib/100_genes/metadata2.csv"),
        Path("/kaggle/input/metadata2/metadata2.csv"),
    ]
    meta_path = first_existing_file(metadata_candidates)
    if meta_path is not None:
        metadata = pd.read_csv(meta_path)
        clade_map_local = dict(zip(metadata["Standardized name"], metadata["Clades"].astype(str).str.strip()))
        clades = [clade_map_local.get(extract_strain_name(sid, gene), "Unknown") for sid in sample_ids]
        return clades, f"metadata2.csv:{meta_path}"

    labels_candidates = [
        ENC_DATASET_ROOT / "labels.txt",
        ENC_DATASET_ROOT / "100_gene_eval" / "labels.txt",
        ENC_DATASET_ROOT / "clades.txt",
        EMBED_ROOT.parent / "labels.txt",
        EMBED_ROOT.parent / "clades.txt",
        Path("/kaggle/input/datasets/mruhaib/100_genes/labels.txt"),
        Path("/kaggle/input/datasets/mruhaib/100_genes/clades.txt"),
    ]
    labels_path = first_existing_file(labels_candidates)
    if labels_path is not None:
        raw_lines = [x.strip() for x in labels_path.read_text(encoding="utf-8").splitlines() if x.strip()]
        if len(raw_lines) == len(sample_ids):
            return raw_lines, f"labels.txt:{labels_path}"
        if len(raw_lines) > 0:
            raise ValueError(
                f"Label file {labels_path} has {len(raw_lines)} lines but gene {gene} has {len(sample_ids)} samples."
            )

    raise FileNotFoundError(
        "Could not locate clade labels. Add metadata2.csv (preferred) or a row-aligned labels.txt/clades.txt file to the Kaggle inputs."
    )

def kmeans_labels(X, k=30):
    X = np.asarray(X)
    if X.ndim != 2:
        return np.zeros(len(X), dtype=int)
    if X.shape[0] < 2 or X.shape[1] == 0:
        return np.zeros(X.shape[0], dtype=int)
    k_eff = max(2, min(k, X.shape[0] - 1))
    km = KMeans(n_clusters=k_eff, random_state=42, n_init=10)
    return km.fit_predict(X)

def safe_silhouette(X, labels):
    X = np.asarray(X)
    if X.ndim != 2 or X.shape[0] < 3 or X.shape[1] == 0:
        return float("nan")
    if len(np.unique(labels)) < 2:
        return float("nan")
    return float(silhouette_score(X, labels))

def center_features(X):
    return X - X.mean(axis=0, keepdims=True)

def linear_cka(X, Y):
    X = center_features(np.asarray(X, dtype=float))
    Y = center_features(np.asarray(Y, dtype=float))
    if X.ndim != 2 or Y.ndim != 2:
        return float("nan")
    if X.shape[0] != Y.shape[0] or X.shape[0] == 0:
        return float("nan")
    xty = X.T @ Y
    hsic = np.linalg.norm(xty, "fro") ** 2
    norm_x = np.linalg.norm(X.T @ X, "fro")
    norm_y = np.linalg.norm(Y.T @ Y, "fro")
    denom = norm_x * norm_y
    if denom <= 0:
        return float("nan")
    return float(hsic / (denom + 1e-12))

def ari_vs_clades(pred_labels, clades):
    clades = np.asarray(clades)
    pred_labels = np.asarray(pred_labels)
    if len(clades) == 0 or len(pred_labels) != len(clades):
        return float("nan")

    valid = clades != "Unknown"
    if valid.sum() < 2:
        return float("nan")

    clades_v = clades[valid]
    pred_v = pred_labels[valid]
    if len(np.unique(clades_v)) < 2:
        return float("nan")

    return float(adjusted_rand_score(clades_v, pred_v))

def read_gene_inputs(gene):
    gene_dir = ENC_DIR / gene
    sample_ids_path = gene_dir / "sample_ids.txt"
    onehot_path = gene_dir / "snp_onehot.npy"
    vcf_path = gene_dir / "vcf_gt_matrix.npy"

    if not (sample_ids_path.exists() and onehot_path.exists() and vcf_path.exists()):
        return None

    sample_ids = [
        x.strip()
        for x in sample_ids_path.read_text(encoding="utf-8").splitlines()
        if x.strip()
    ]
    onehot = np.load(onehot_path)
    vcf_cols = np.load(vcf_path)
    return sample_ids, onehot, vcf_cols

def save_plot(fig, out_path):
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)

def tick_step_for_n(n):
    if n <= 30:
        return 1
    if n <= 60:
        return 2
    if n <= 120:
        return 5
    return max(1, n // 24)

In [ ]:
# -----------------------------
# Metadata and gene inventory
# -----------------------------
if META_PATH is not None:
    metadata = pd.read_csv(META_PATH)
    clade_map = dict(zip(metadata["Standardized name"], metadata["Clades"].astype(str).str.strip()))
else:
    metadata = pd.DataFrame()
    clade_map = {}

genes = sorted([d.name for d in ENC_DIR.iterdir() if d.is_dir()])
if len(genes) == 0:
    raise RuntimeError(f"No gene directories found in {ENC_DIR}")

gene_info = pd.DataFrame({"gene": genes})
gene_info["seq_len"] = gene_info["gene"].apply(extract_gene_length)
gene_info = gene_info.sort_values(["seq_len", "gene"], na_position="last").reset_index(drop=True)
genes_sorted = gene_info["gene"].tolist()

print(f"Found {len(genes_sorted)} genes in encodings dataset")
print(f"metadata rows: {len(metadata)}")
if gene_info["seq_len"].notna().any():
    print(f"Sequence length range: {int(gene_info['seq_len'].min())} to {int(gene_info['seq_len'].max())}")
else:
    print("Could not parse sequence lengths from gene names.")

display(gene_info.head(15))

In [ ]:
# -----------------------------
# Compute metrics for all genes and models
# -----------------------------
rows = []

for gi, gene in enumerate(genes_sorted, start=1):
    loaded = read_gene_inputs(gene)
    if loaded is None:
        print(f"[skip] Missing encoding files for {gene}")
        continue

    sample_ids, onehot, vcf_cols = loaded
    n = len(sample_ids)
    seq_len = extract_gene_length(gene)

    clades, clade_source = load_clade_labels_for_gene(sample_ids, gene)

    embedding_files = find_embedding_files_for_gene(EMBED_ROOT, gene)
    if len(embedding_files) == 0:
        print(f"[skip] No embeddings found for {gene}")
        continue

    onehot_labels = kmeans_labels(onehot, k=K_CLUSTERS)
    vcf_labels = kmeans_labels(vcf_cols, k=K_CLUSTERS)

    ari_onehot_vs_clades = ari_vs_clades(onehot_labels, clades)
    ari_vcf_vs_clades = ari_vs_clades(vcf_labels, clades)
    ari_onehot_vs_vcf = float(adjusted_rand_score(onehot_labels, vcf_labels))

    sil_onehot = safe_silhouette(onehot, onehot_labels)
    sil_vcf = safe_silhouette(vcf_cols, vcf_labels)

    cka_onehot_vs_vcf = linear_cka(onehot, vcf_cols)

    usable_models = 0
    for model_short, emb_file in embedding_files.items():
        try:
            glm = load_glm_embeddings(emb_file)
        except Exception as e:
            print(f"[skip] {gene} | {model_short}: parse error in {emb_file.name} -> {e}")
            continue

        if glm.shape[0] != n:
            print(f"[skip] {gene} | {model_short}: sample mismatch {glm.shape[0]} vs {n}")
            continue

        glm_labels = kmeans_labels(glm, k=K_CLUSTERS)
        sil_glm = safe_silhouette(glm, glm_labels)

        row = {
            "gene": gene,
            "seq_len": seq_len,
            "model": model_short,
            "clade_source": clade_source,
            "n_samples": int(n),
            "glm_dim": int(glm.shape[1]),
            "onehot_dim": int(onehot.shape[1]),
            "vcf_dim": int(vcf_cols.shape[1]),
            "ari_glm_vs_clades": ari_vs_clades(glm_labels, clades),
            "ari_onehot_vs_clades": ari_onehot_vs_clades,
            "ari_vcf_vs_clades": ari_vcf_vs_clades,
            "ari_glm_vs_onehot": float(adjusted_rand_score(glm_labels, onehot_labels)),
            "ari_glm_vs_vcf": float(adjusted_rand_score(glm_labels, vcf_labels)),
            "ari_onehot_vs_vcf": ari_onehot_vs_vcf,
            "sil_glm": sil_glm,
            "sil_onehot": sil_onehot,
            "sil_vcf": sil_vcf,
            "cka_glm_vs_onehot": linear_cka(glm, onehot),
            "cka_glm_vs_vcf": linear_cka(glm, vcf_cols),
            "cka_onehot_vs_vcf": cka_onehot_vs_vcf,
        }
        rows.append(row)
        usable_models += 1

    if gi % 10 == 0 or gi == len(genes_sorted):
        print(f"Processed {gi}/{len(genes_sorted)} genes | usable models this gene: {usable_models}")

results_df = pd.DataFrame(rows)
if len(results_df) == 0:
    raise RuntimeError("No metric rows were produced. Check paths, file naming, and label inputs.")

results_path = RESULTS_DIR / "glm_oh_vcf_ari_cka_modelwise_source.csv"
results_df.to_csv(results_path, index=False)

print(f"Saved results to: {results_path}")
print(f"Rows: {len(results_df)}")
print(results_df[["gene", "model", "ari_glm_vs_clades", "ari_onehot_vs_clades", "ari_vcf_vs_clades"]].head())
display(results_df.head())

In [ ]:
# -----------------------------
# Prepare model-wise plotting dataframe
# -----------------------------
results_path = RESULTS_DIR / "glm_oh_vcf_ari_cka_modelwise_source.csv"
if results_path.exists():
    plot_df = pd.read_csv(results_path)
else:
    plot_df = results_df.copy()

if "seq_len" not in plot_df.columns:
    plot_df["seq_len"] = plot_df["gene"].apply(extract_gene_length)

gene_order = (
    plot_df[["gene", "seq_len"]]
    .drop_duplicates()
    .sort_values(["seq_len", "gene"], na_position="last")
    ["gene"]
    .tolist()
)

models_sorted = sorted(plot_df["model"].dropna().unique().tolist())

print(f"Loaded rows: {len(plot_df)}")
print(f"Models found: {len(models_sorted)}")
print(f"Genes in order: {len(gene_order)}")
display(plot_df.head())

In [ ]:
# -----------------------------
# Model-wise plots for all 5 analyses
# -----------------------------
ARI_DIR = PLOTS_DIR / "01_individual_ari"
SIL_DIR = PLOTS_DIR / "02_individual_silhouette"
PAIR_ARI_DIR = PLOTS_DIR / "03_pairwise_ari"
CKA_OH_DIR = PLOTS_DIR / "04_cka_glm_vs_onehot"
CKA_VCF_DIR = PLOTS_DIR / "05_cka_glm_vs_vcf"

for p in [ARI_DIR, SIL_DIR, PAIR_ARI_DIR, CKA_OH_DIR, CKA_VCF_DIR]:
    p.mkdir(parents=True, exist_ok=True)

for model in models_sorted:
    g = plot_df[plot_df["model"] == model].copy()
    g = g.set_index("gene").reindex(gene_order).reset_index()

    labels = g["gene"].tolist()
    x = np.arange(len(labels))
    step = tick_step_for_n(len(labels))
    xticks = x[::step]
    xticklabels = [labels[i] for i in range(0, len(labels), step)]

    model_slug = safe_filename(model)

    # 1) Individual ARI
    fig, ax = plt.subplots(figsize=(24, 7))
    ax.plot(x, g["ari_glm_vs_clades"], marker="o", linewidth=2, label="ARI: gLM vs clades")
    ax.plot(x, g["ari_onehot_vs_clades"], linestyle="--", linewidth=2, label="ARI: one-hot vs clades")
    ax.plot(x, g["ari_vcf_vs_clades"], linestyle="--", linewidth=2, label="ARI: VCF vs clades")
    ax.set_title(f"{model} | Individual ARI across genes (sorted by sequence length)")
    ax.set_xlabel("Gene")
    ax.set_ylabel("ARI")
    ax.set_ylim(0, 1)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=75, ha="right")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    save_plot(fig, ARI_DIR / f"{model_slug}_01_individual_ari.png")

    # 2) Individual silhouette
    fig, ax = plt.subplots(figsize=(24, 7))
    ax.plot(x, g["sil_glm"], marker="o", linewidth=2, label="Silhouette: gLM")
    ax.plot(x, g["sil_onehot"], linestyle="--", linewidth=2, label="Silhouette: one-hot")
    ax.plot(x, g["sil_vcf"], linestyle="--", linewidth=2, label="Silhouette: VCF")
    ax.set_title(f"{model} | Individual silhouette across genes (sorted by sequence length)")
    ax.set_xlabel("Gene")
    ax.set_ylabel("Silhouette")
    ax.set_ylim(0, 1)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=75, ha="right")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    save_plot(fig, SIL_DIR / f"{model_slug}_02_individual_silhouette.png")

    # 3) Pairwise ARI
    fig, ax = plt.subplots(figsize=(24, 7))
    ax.plot(x, g["ari_glm_vs_onehot"], marker="o", linewidth=2, label="ARI: gLM vs one-hot")
    ax.plot(x, g["ari_glm_vs_vcf"], marker="o", linewidth=2, label="ARI: gLM vs VCF")
    ax.plot(x, g["ari_glm_vs_clades"], marker="o", linewidth=2, label="ARI: gLM vs clades")
    ax.plot(x, g["ari_onehot_vs_clades"], linestyle="--", linewidth=2, label="ARI baseline: one-hot vs clades")
    ax.plot(x, g["ari_vcf_vs_clades"], linestyle="--", linewidth=2, label="ARI baseline: VCF vs clades")
    ax.set_title(f"{model} | Pairwise ARI across genes (sorted by sequence length)")
    ax.set_xlabel("Gene")
    ax.set_ylabel("ARI")
    ax.set_ylim(0, 1)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=75, ha="right")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    save_plot(fig, PAIR_ARI_DIR / f"{model_slug}_03_pairwise_ari.png")

    # 4) CKA gLM vs one-hot
    fig, ax = plt.subplots(figsize=(24, 7))
    ax.plot(x, g["cka_glm_vs_onehot"], marker="o", linewidth=2, label="CKA: gLM vs one-hot")
    ax.plot(x, g["cka_onehot_vs_vcf"], linestyle="--", linewidth=2, label="CKA baseline: one-hot vs VCF")
    ax.set_title(f"{model} | CKA gLM vs one-hot across genes (sorted by sequence length)")
    ax.set_xlabel("Gene")
    ax.set_ylabel("Linear CKA")
    ax.set_ylim(0, 1)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=75, ha="right")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    save_plot(fig, CKA_OH_DIR / f"{model_slug}_04_cka_glm_vs_onehot.png")

    # 5) CKA gLM vs VCF
    fig, ax = plt.subplots(figsize=(24, 7))
    ax.plot(x, g["cka_glm_vs_vcf"], marker="o", linewidth=2, label="CKA: gLM vs VCF")
    ax.plot(x, g["cka_onehot_vs_vcf"], linestyle="--", linewidth=2, label="CKA baseline: one-hot vs VCF")
    ax.set_title(f"{model} | CKA gLM vs VCF across genes (sorted by sequence length)")
    ax.set_xlabel("Gene")
    ax.set_ylabel("Linear CKA")
    ax.set_ylim(0, 1)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=75, ha="right")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    save_plot(fig, CKA_VCF_DIR / f"{model_slug}_05_cka_glm_vs_vcf.png")

print(f"Saved model-wise plots under: {PLOTS_DIR}")

In [ ]:
# -----------------------------
# Model comparison averages across all genes
# -----------------------------
summary_df = (
    plot_df.groupby("model", as_index=False)
    .agg(
        n_genes=("gene", "nunique"),
        analysis1_individual_ari_mean=("ari_glm_vs_clades", "mean"),
        analysis1_individual_ari_std=("ari_glm_vs_clades", "std"),
        analysis2_silhouette_mean=("sil_glm", "mean"),
        analysis2_silhouette_std=("sil_glm", "std"),
        analysis3_pairwise_ari_glm_onehot_mean=("ari_glm_vs_onehot", "mean"),
        analysis3_pairwise_ari_glm_vcf_mean=("ari_glm_vs_vcf", "mean"),
        analysis4_cka_glm_vs_onehot_mean=("cka_glm_vs_onehot", "mean"),
        analysis5_cka_glm_vs_vcf_mean=("cka_glm_vs_vcf", "mean"),
        baseline_ari_onehot_vs_clades_mean=("ari_onehot_vs_clades", "mean"),
        baseline_ari_vcf_vs_clades_mean=("ari_vcf_vs_clades", "mean"),
        baseline_cka_onehot_vs_vcf_mean=("cka_onehot_vs_vcf", "mean"),
    )
)

summary_df["analysis3_pairwise_ari_mean"] = (
    summary_df["analysis3_pairwise_ari_glm_onehot_mean"]
    + summary_df["analysis3_pairwise_ari_glm_vcf_mean"]
) / 2.0

summary_df = summary_df.sort_values("analysis1_individual_ari_mean", ascending=False).reset_index(drop=True)

summary_path = RESULTS_DIR / "model_comparison_summary_5_analyses.csv"
summary_df.to_csv(summary_path, index=False)

display(summary_df)
print(f"Saved summary table: {summary_path}")

comparison_plot_path = RESULTS_DIR / "model_comparison_summary_5_analyses.png"
metrics_for_plot = [
    ("analysis1_individual_ari_mean", "1) Mean Individual ARI"),
    ("analysis2_silhouette_mean", "2) Mean Silhouette"),
    ("analysis3_pairwise_ari_mean", "3) Mean Pairwise ARI"),
    ("analysis4_cka_glm_vs_onehot_mean", "4) Mean CKA gLM vs one-hot"),
    ("analysis5_cka_glm_vs_vcf_mean", "5) Mean CKA gLM vs VCF"),
]

fig, axes = plt.subplots(len(metrics_for_plot), 1, figsize=(18, 26))
for ax, (col, title) in zip(axes, metrics_for_plot):
    s = summary_df.sort_values(col, ascending=False)
    ax.bar(s["model"], s[col], color="tab:blue")
    ax.set_title(title)
    ax.set_xlabel("Model")
    ax.set_ylabel("Mean score")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=75)
    ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
fig.savefig(comparison_plot_path, dpi=180)
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

print(f"Saved comparison figure: {comparison_plot_path}")